<a href="https://colab.research.google.com/github/Shahana023/cse-resources/blob/main/autonomous_vit/AV_adversarial_defense_proposal_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A Robust and Explainable Defense for Autonomous-Vehicle Perception under *Adaptive* Physical Adversarial Attacks
### PhD Research Proposal — Motivating Demonstration
**Area:** Adversarial machine learning · autonomous-vehicle perception · AI security

---

### 1. Problem statement
Autonomous-vehicle (AV) perception depends on deep object detectors that can be fooled by **adversarial attacks** — small, deliberate changes to what the camera sees that make the detector miss real objects, so a pedestrian or vehicle effectively "disappears." This is a safety-critical failure mode.

### 2. Research gap
Adaptive-attacker evaluation of AV patch defenses already exists (e.g., the ICCV 2025 benchmark, arXiv:2508.00649, which adaptively re-evaluated 11 detector defenses including a transformer detector; and AV-specific defenses such as PhySense, CCS 2024). What remains genuinely unfilled is a defense that is **simultaneously (i) evaluated against an adaptive attacker (Athalye et al., 2018), (ii) genuinely _explainable_ (a human-auditable rationale, not just an internal attribution map), and (iii) validated on AV transformer-family detectors (DETR).** No existing work occupies that intersection.

### 3. Proposed contribution
A defense for AV object detectors that is **adaptively robust *and* genuinely explainable**, demonstrated on transformer-family (DETR) detectors under real-time constraints. The detector architecture is a design choice; the contribution is pairing genuine explainability with adaptive-attack rigour in the AV detection setting.

### 4. What this notebook demonstrates
On a real **KITTI** driving frame, using a pretrained **DETR** detector:
1. DETR detects the vehicles and objects normally.
2. A **near-invisible** digital adversarial perturbation makes DETR detect **nothing** — showing how fragile camera perception is.
3. A simple **high-frequency probe** flags *this* attack cheaply (a first, honest step toward a defense).

> **Why a digital attack here?** It is reliable and reproducible, so it cleanly illustrates the vulnerability. The proposed *research* targets the harder, realistic case: **physical** patch attacks, and a defense that stays robust against an **adaptive** attacker and **explains** its decisions. See the honest scope note at the end.

## Step 0 — Setup

In [ ]:
!pip install -q transformers timm torch torchvision pillow matplotlib requests hf_transfer

### (Optional) Hugging Face token — works on Colab *and* Kaggle
The model is public, so **no token is needed** — it only speeds the download. Options (or skip):
- **Any platform:** paste your token into the `HF_TOKEN = ""` line below *(don't save/share the notebook with it filled in)*.
- **Colab:** leave it blank — a paste box appears (or use the 🔑 Secrets panel, name `HF_TOKEN`).
- **Kaggle:** add it under **Add-ons ▸ Secrets** (name `HF_TOKEN`).

In [ ]:
import os
HF_TOKEN = ""   # optional: paste your token here, else leave ""
_tok = HF_TOKEN.strip()
if not _tok:
    try:
        from google.colab import userdata; _tok = userdata.get("HF_TOKEN") or ""
    except Exception: pass
if not _tok:
    try:
        from kaggle_secrets import UserSecretsClient; _tok = UserSecretsClient().get_secret("HF_TOKEN") or ""
    except Exception: pass
if not _tok:
    try:
        import getpass; _tok = getpass.getpass("Paste HF token + Enter (or just Enter to skip): ").strip()
    except Exception: pass
if _tok: os.environ["HF_TOKEN"] = _tok; print("HF token set.")
else: print("No token — continuing without one (the public model still works).")

## Step 1 — Load a pretrained DETR detector

In [ ]:
import time, numpy as np, torch, torch.nn.functional as F, requests
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from transformers import DetrImageProcessor, DetrForObjectDetection
try:
    import hf_transfer; os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except Exception: pass

device = "cuda" if torch.cuda.is_available() else "cpu"
IMG = (320, 1024)                 # KITTI-friendly (H, W)
EPS = 8/255; STEPS = 30; ALPHA = 2/255   # attack budget (8/255 is ~invisible)

proc = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50").to(device).eval()
for p in model.parameters(): p.requires_grad_(False)
MEAN = torch.tensor(proc.image_mean, device=device).view(1,3,1,1)
STD  = torch.tensor(proc.image_std,  device=device).view(1,3,1,1)
NO = model.config.num_labels; COCO = model.config.id2label
norm = lambda x: (x - MEAN) / STD
print("loaded DETR on", device)

## Step 2 — A real driving scene (KITTI)

In [ ]:
r = requests.get("https://raw.githubusercontent.com/open-mmlab/mmdetection3d/main/demo/data/kitti/000008.png",
                 timeout=30, headers={"User-Agent": "Mozilla/5.0"})
pil = Image.open(BytesIO(r.content)).convert("RGB").resize((IMG[1], IMG[0]))
img = torch.from_numpy(np.array(pil).astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(device)
# Use your own KITTI/BDD100K frame instead:
# pil = Image.open("/content/your_frame.png").convert("RGB").resize((IMG[1], IMG[0]))
# img = torch.from_numpy(np.array(pil).astype(np.float32)/255).permute(2,0,1).unsqueeze(0).to(device)
print("image:", tuple(img.shape))

## Step 3 — Baseline detection (no attack)

In [ ]:
def detect(x, th=0.7):
    with torch.no_grad(): out = model(pixel_values=norm(x))
    return proc.post_process_object_detection(out, target_sizes=torch.tensor([IMG], device=device), threshold=th)[0]
def show(x, res, title, ax):
    ax.imshow(np.clip(x.squeeze(0).permute(1,2,0).detach().cpu().numpy(), 0, 1))
    for s,l,b in zip(res["scores"], res["labels"], res["boxes"]):
        x0,y0,x1,y1 = b.tolist()
        ax.add_patch(mpatches.Rectangle((x0,y0),x1-x0,y1-y0,lw=2,edgecolor="lime",facecolor="none"))
        ax.text(x0,y0-4,f'{COCO[l.item()]}:{s:.2f}',color="lime",fontsize=7,bbox=dict(facecolor="black",alpha=0.5,pad=1))
    ax.set_title(title); ax.axis("off")

base = detect(img); n0 = len(base["scores"])
fig, ax = plt.subplots(figsize=(14,4)); show(img, base, f"DETR detects {n0} objects", ax); plt.show()
print("baseline objects:", n0)

## Step 4 — A near-invisible digital attack
We nudge the image pixels with a tiny, bounded perturbation (PGD) so DETR stops seeing objects.
The change per pixel is capped at 8/255 ≈ 0.03 — imperceptible to a person.

In [ ]:
def fg(x): return 1 - model(pixel_values=norm(x)).logits.softmax(-1)[..., NO]
adv = img.clone()
for i in range(STEPS):
    adv.requires_grad_(True)
    g = torch.autograd.grad(fg(adv).topk(min(20, fg(adv).shape[1]), dim=1).values.mean(), adv)[0]
    adv = (adv - ALPHA*g.sign()).detach()
    adv = torch.min(torch.max(adv, img-EPS), img+EPS).clamp(0, 1)
atk = detect(adv); nA = len(atk["scores"]); linf = (adv-img).abs().max().item()

fig, ax = plt.subplots(1, 2, figsize=(20, 5))
show(img, base, f"Clean — DETR sees {n0} objects", ax[0])
show(adv, atk, f"Attacked — DETR sees {nA} objects", ax[1])
plt.tight_layout(); plt.show()
print(f"objects {n0} -> {nA}   (largest per-pixel change = {linf:.3f} of 1.0 — invisible to a human)")

## Step 5 — The perturbation is essentially invisible

In [ ]:
pert = (adv-img).squeeze(0).permute(1,2,0).detach().cpu().numpy()
fig, ax = plt.subplots(figsize=(14,4))
ax.imshow(np.clip((pert-pert.min())/(pert.max()-pert.min()+1e-9), 0, 1)); ax.axis("off")
ax.set_title(f"The perturbation, contrast-amplified ~30x to be visible. Real magnitude: {linf:.3f} per pixel.")
plt.show()

## Step 6 — A first, honest defense probe
Adversarial perturbations inject high-frequency noise. We measure high-frequency (Laplacian)
energy and compare clean vs attacked. **The verdict below is computed from the actual numbers,
not assumed.**

In [ ]:
def hf_energy(x):
    k = torch.tensor([[0,-1,0],[-1,4,-1],[0,-1,0]], dtype=torch.float32, device=device).view(1,1,3,3)
    return F.conv2d(x.mean(1, keepdim=True), k, padding=1).abs().mean().item()

e0, eA = hf_energy(img), hf_energy(adv); ratio = eA/e0
flagged = ratio > 1.2
print(f"high-frequency energy — clean: {e0:.4f}   attacked: {eA:.4f}   ratio: {ratio:.2f}x")
print(f"VERDICT (computed): the attacked image has {100*(ratio-1):.0f}% more high-frequency energy -> "
      + ("probe FLAGS it as adversarial." if flagged else "probe does NOT separate it (needs a better defense)."))

## Step 7 — Efficiency (real-time constraint)

In [ ]:
def bench(fn, n=10):
    if device=="cuda": torch.cuda.synchronize()
    t=time.time()
    for _ in range(n): fn()
    if device=="cuda": torch.cuda.synchronize()
    return (time.time()-t)/n*1000
base_ms = bench(lambda: detect(img)); def_ms = bench(lambda: hf_energy(img), 50)
print(f"detector       : {base_ms:6.1f} ms/frame ({1000/base_ms:5.1f} FPS)")
print(f"defense probe  : {def_ms:6.2f} ms/frame  (+{def_ms:.2f} ms overhead)")

## Step 8 — What I propose, and honest scope

**Objective.** Design a defense for AV object detectors (DETR-family) that is **adaptively
robust *and* genuinely explainable**, validated under real-time constraints on real driving
data (KITTI, BDD100K).

1. **Benchmark (first, publishable result).** Extend the ICCV 2025 adaptive-evaluation
   methodology (arXiv:2508.00649) into the AV / DETR setting, and test whether defenses
   marketed as *explainable* (X-Detect, AdvPatchXAI) or *AV-adaptive* (PhySense) keep **both**
   robustness and usable explanations once the attack is adapted to them.
2. **Defense design.** Build the defense that occupies the empty cell — adaptive-robust,
   explainable, DETR-family, AV.
3. **Guarantees (stretch).** Pursue *certified* robustness for transformer detectors, an
   open sub-problem (existing certified patch defenses assume CNN receptive fields).

**Honest scope of THIS notebook (state these to your guide):**
- The attack here is **digital** (perturbs image pixels) and **non-adaptive**. It reliably
  shows the *vulnerability*; a real-world attack would be a **physical** patch, which is
  harder to build and is part of the proposed research.
- The high-frequency probe is a **first heuristic**. It catches this non-adaptive attack, but
  an **adaptive** attacker that knows the probe could suppress the high-frequency signal —
  which is exactly why the thesis centres on *adaptive-robust* and *explainable* defense.
- This is a single-image proof-of-concept, not a dataset-scale result (no mAP / attack-success
  rate over KITTI/BDD100K).

## References
- A. Athalye, N. Carlini, D. Wagner. *Obfuscated Gradients Give a False Sense of Security.* ICML 2018.
- *Revisiting Adversarial Patch Defenses on Object Detectors.* ICCV 2025. (arXiv:2508.00649)
- *NutNet: Real-Time Defense against Diverse Adversarial Patches for Object Detectors.* ACM CCS 2024. (arXiv:2406.10285)
- *PhySense: Defending Physically Realizable Attacks for Autonomous Systems via Consistency Reasoning.* ACM CCS 2024.
- *AdvPatchXAI: A Unified, Resilient, and Explainable Adversarial Patch Detector.* CVPR 2025.
- *X-Detect: Explainable Adversarial Patch Detection for Object Detectors.* Machine Learning (Springer), 2024. (arXiv:2306.08422)